[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/092.%20The%20Location-Routing%20Problem%20%28LRP%29/P92-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 92. The Location-Routing Problem (LRP)
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Fixed costs for opening depots are known
- Customer demands are deterministic
- Vehicle capacity is limited and identical for all vehicles
- Travel costs between all locations are known
- Each customer must be served by exactly one depot
- Routes must start and end at the same depot

### Approach (step-by-step)
1. **Problem Definition**: Define sets, parameters, and decision variables
2. **Mathematical Model**: Formulate the Mixed-Integer Programming (MIP) model
3. **Concrete Example**: Implement the 2-depot, 3-customer example from the source
4. **Solution Method**: Use pulp solver with fallback enumeration
5. **Visualization**: Create network visualization of the solution
6. **Sensitivity Analysis**: Analyze parameter impacts on the solution

### What to look for in the results
- Which depots are opened (binary decisions)
- Customer-depot assignments
- Vehicle routes for each depot
- Total cost breakdown (fixed + variable costs)
- Capacity utilization of vehicles

### Concrete example (from the source)
- **Depots**: 2 potential locations (J₁, J₂)
- **Customers**: 3 customers (C₁, C₂, C₃)
- **Depot Costs**: f₁=100, f₂=120
- **Demands**: d₁=10, d₂=15, d₃=20
- **Vehicle Capacity**: Q=30
- **Expected Solution**: Open depot J₁, use 2 vehicles to serve all customers

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import itertools
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class LRPInstance:
    """Data class for Location-Routing Problem instance"""
    # Sets
    customers: List[int]  # Customer indices
    depots: List[int]     # Depot indices
    vehicles: List[int]   # Vehicle indices
    
    # Parameters
    depot_costs: Dict[int, float]      # Fixed cost to open each depot
    demands: Dict[int, float]          # Customer demands
    vehicle_capacity: float            # Capacity of each vehicle
    travel_costs: Dict[Tuple[int, int], float]  # Travel costs between nodes
    
    def get_all_nodes(self):
        """Return all nodes (customers + depots)"""
        return self.customers + self.depots

# Create the concrete example from the source
def create_concrete_example():
    """Create the 2-depot, 3-customer example"""
    
    # Define sets
    customers = [1, 2, 3]  # C1, C2, C3
    depots = [4, 5]         # J1, J2 (using 4,5 to avoid confusion)
    vehicles = [1, 2]       # Two vehicles available
    
    # Define parameters
    depot_costs = {4: 100, 5: 120}  # f1=100, f2=120
    demands = {1: 10, 2: 15, 3: 20}  # d1=10, d2=15, d3=20
    vehicle_capacity = 30  # Q=30
    
    # Define travel costs (symmetric)
    # Using reasonable distances for visualization
    travel_costs = {
        # Between customers
        (1, 2): 15, (2, 1): 15,
        (1, 3): 20, (3, 1): 20,
        (2, 3): 18, (3, 2): 18,
        # Between depots and customers
        (4, 1): 12, (1, 4): 12,  # J1 to C1
        (4, 2): 14, (2, 4): 14,  # J1 to C2
        (4, 3): 25, (3, 4): 25,  # J1 to C3
        (5, 1): 18, (1, 5): 18,  # J2 to C1
        (5, 2): 16, (2, 5): 16,  # J2 to C2
        (5, 3): 22, (3, 5): 22,  # J2 to C3
        # Between depots
        (4, 5): 30, (5, 4): 30,
        # Self-loops (zero cost)
        (1, 1): 0, (2, 2): 0, (3, 3): 0,
        (4, 4): 0, (5, 5): 0
    }
    
    return LRPInstance(
        customers=customers,
        depots=depots,
        vehicles=vehicles,
        depot_costs=depot_costs,
        demands=demands,
        vehicle_capacity=vehicle_capacity,
        travel_costs=travel_costs
    )

# Create the instance
instance = create_concrete_example()
print(f"LRP Instance created:")
print(f"Customers: {instance.customers}")
print(f"Potential Depots: {instance.depots}")
print(f"Vehicles: {instance.vehicles}")
print(f"Depot Costs: {instance.depot_costs}")
print(f"Customer Demands: {instance.demands}")
print(f"Vehicle Capacity: {instance.vehicle_capacity}")

LRP Instance created:
Customers: [1, 2, 3]
Potential Depots: [4, 5]
Vehicles: [1, 2]
Depot Costs: {4: 100, 5: 120}
Customer Demands: {1: 10, 2: 15, 3: 20}
Vehicle Capacity: 30


In [ ]:
def solve_lrp_mip(instance: LRPInstance):
    """Solve the Location-Routing Problem using Mixed-Integer Programming"""
    
    # Create the problem
    prob = LpProblem("Location_Routing_Problem", LpMinimize)
    
    # Decision variables
    # y_j: 1 if depot j is opened, 0 otherwise
    y = {j: LpVariable(f"y_{j}", cat='Binary') for j in instance.depots}
    
    # z_ij: 1 if customer i is assigned to depot j, 0 otherwise
    z = {(i, j): LpVariable(f"z_{i}_{j}", cat='Binary') 
         for i in instance.customers for j in instance.depots}
    
    # x_uvk: 1 if vehicle k travels from u to v, 0 otherwise
    x = {(u, v, k): LpVariable(f"x_{u}_{v}_{k}", cat='Binary')
         for u in instance.get_all_nodes() for v in instance.get_all_nodes() 
         for k in instance.vehicles}
    
    # Objective function: minimize total cost (fixed depot costs + routing costs)
    prob += (
        lpSum([instance.depot_costs[j] * y[j] for j in instance.depots]) +
        lpSum([instance.travel_costs[(u, v)] * x[(u, v, k)] 
               for u in instance.get_all_nodes() for v in instance.get_all_nodes() 
               for k in instance.vehicles])
    ), "Total_Cost"
    
    # Constraints
    
    # 1. Each customer must be assigned to exactly one depot
    for i in instance.customers:
        prob += lpSum([z[(i, j)] for j in instance.depots]) == 1, f"Assign_Customer_{i}"
    
    # 2. Customers can only be assigned to open depots
    for i in instance.customers:
        for j in instance.depots:
            prob += z[(i, j)] <= y[j], f"Link_Assign_Open_{i}_{j}"
    
    # 3. Flow conservation for each vehicle
    for k in instance.vehicles:
        for u in instance.get_all_nodes():
            prob += (
                lpSum([x[(u, v, k)] for v in instance.get_all_nodes()]) -
                lpSum([x[(v, u, k)] for v in instance.get_all_nodes()]) == 0
            ), f"Flow_Conservation_{u}_{k}"
    
    # 4. Each customer must be visited exactly once by exactly one vehicle
    for i in instance.customers:
        prob += lpSum([x[(u, i, k)] for u in instance.get_all_nodes() 
                     for k in instance.vehicles]) == 1, f"Visit_Customer_{i}"
    
    # 5. Vehicle capacity constraints (simplified for this small example)
    for k in instance.vehicles:
        prob += (
            lpSum([instance.demands[i] * lpSum([x[(u, i, k)] for u in instance.get_all_nodes()]) 
                   for i in instance.customers]) <= instance.vehicle_capacity
        ), f"Vehicle_Capacity_{k}"
    
    # 6. Each vehicle starts from at most one depot
    for j in instance.depots:
        for k in instance.vehicles:
            prob += lpSum([x[(j, v, k)] for v in instance.get_all_nodes()]) <= 1, f"Start_Depot_{j}_{k}"
    
    # 7. Subtour elimination (simplified MTZ constraints for small instance)
    # Add auxiliary variables for MTZ
    u = {i: LpVariable(f"u_{i}", lowBound=0) for i in instance.customers}
    
    for i in instance.customers:
        for j in instance.customers:
            if i != j:
                for k in instance.vehicles:
                    prob += u[i] - u[j] + len(instance.customers) * x[(i, j, k)] <= len(instance.customers) - 1
    
    # Solve the problem
    print("Solving LRP using MIP...")
    prob.solve(PULP_CBC_CMD(msg=False))
    
    return prob, y, z, x

# Solve the problem
prob, y_vars, z_vars, x_vars = solve_lrp_mip(instance)

# Check solution status
print(f"\nSolution Status: {LpStatus[prob.status]}")
print(f"Objective Value (Total Cost): ${prob.objective.value():.2f}")

Solving LRP using MIP...
Generation 5: Best fitness = 4101.11

Solution Status: Optimal
Objective Value (Total Cost): $100.00


In [ ]:
def extract_solution(instance, prob, y_vars, z_vars, x_vars):
    """Extract and format the solution"""
    
    if prob.status != 1:  # Not optimal
        return None, None, None
    
    # Extract depot decisions
    opened_depots = [j for j in instance.depots if y_vars[j].value() == 1]
    
    # Extract customer assignments
    assignments = {}
    for i in instance.customers:
        for j in instance.depots:
            if z_vars[(i, j)].value() == 1:
                assignments[i] = j
                break
    
    # Extract routes for each vehicle
    routes = {}
    for k in instance.vehicles:
        route = []
        # Find edges used by this vehicle
        edges = [(u, v) for u in instance.get_all_nodes() for v in instance.get_all_nodes() 
                if x_vars[(u, v, k)].value() == 1]
        
        if edges:
            # Build route starting from a depot
            start_depot = None
            for u, v in edges:
                if u in instance.depots:
                    start_depot = u
                    break
            
            if start_depot:
                current = start_depot
                route = [current]
                visited = set([current])
                
                # Follow the path
                while True:
                    found_next = False
                    for u, v in edges:
                        if u == current and v not in visited:
                            current = v
                            route.append(current)
                            visited.add(current)
                            found_next = True
                            break
                    
                    # Stop if we're back at a depot or can't find next node
                    if not found_next or current in instance.depots:
                        break
                
                routes[k] = route
    
    return opened_depots, assignments, routes

# Extract solution
opened_depots, assignments, routes = extract_solution(instance, prob, y_vars, z_vars, x_vars)

print("Solution Details:")
print(f"Opened Depots: {opened_depots}")
print(f"Customer Assignments: {assignments}")
print(f"Routes: {routes}")

Solution Details:
Opened Depots: [4]
Customer Assignments: {1: 4, 2: 4, 3: 4}
Routes: {}


In [ ]:
def calculate_solution_cost(instance, opened_depots, assignments, routes):
    """Calculate the total cost of the solution"""
    
    # Fixed depot costs
    fixed_cost = sum(instance.depot_costs[j] for j in opened_depots)
    
    # Routing costs
    routing_cost = 0
    for k, route in routes.items():
        for i in range(len(route) - 1):
            u, v = route[i], route[i + 1]
            routing_cost += instance.travel_costs[(u, v)]
    
    total_cost = fixed_cost + routing_cost
    return total_cost, fixed_cost, routing_cost

# Calculate costs
total_cost, fixed_cost, routing_cost = calculate_solution_cost(
    instance, opened_depots, assignments, routes
)

print("Cost Breakdown:")
print(f"Fixed Depot Costs: ${fixed_cost:.2f}")
print(f"Routing Costs: ${routing_cost:.2f}")
print(f"Total Cost: ${total_cost:.2f}")

# Calculate capacity utilization
print("\nVehicle Capacity Utilization:")
for k, route in routes.items():
    route_demand = sum(instance.demands[i] for i in route if i in instance.customers)
    utilization = (route_demand / instance.vehicle_capacity) * 100
    print(f"Vehicle {k}: {route_demand}/{instance.vehicle_capacity} ({utilization:.1f}%)")

Cost Breakdown:
Fixed Depot Costs: $100.00
Routing Costs: $0.00
Total Cost: $100.00

Vehicle Capacity Utilization:


In [ ]:
def visualize_lrp_solution(instance, opened_depots, assignments, routes):
    """Visualize the LRP solution"""
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Define positions for visualization
    positions = {
        # Customer positions (clustered in different areas)
        1: (2, 8),   # C1
        2: (4, 6),   # C2
        3: (8, 7),   # C3
        # Depot positions
        4: (3, 3),   # J1
        5: (7, 3),   # J2
    }
    
    # Plot all nodes
    for node, pos in positions.items():
        if node in instance.customers:
            # Plot customers
            ax.scatter(pos[0], pos[1], s=300, c='lightblue', 
                      edgecolors='navy', linewidth=2, zorder=5)
            ax.annotate(f'C{node}\n(d={instance.demands[node]})', 
                       xy=pos, xytext=(pos[0]+0.3, pos[1]+0.3),
                       fontsize=10, fontweight='bold')
        elif node in instance.depots:
            # Plot depots
            color = 'lightgreen' if node in opened_depots else 'lightgray'
            ax.scatter(pos[0], pos[1], s=400, c=color, 
                      edgecolors='darkgreen', linewidth=2, zorder=5)
            status = "OPEN" if node in opened_depots else "CLOSED"
            cost = instance.depot_costs[node]
            ax.annotate(f'J{node-3}\n({status})\n(${cost})', 
                       xy=pos, xytext=(pos[0]-0.8, pos[1]-1.2),
                       fontsize=10, fontweight='bold')
    
    # Plot routes
    colors = ['red', 'blue', 'green', 'orange']
    for k, route in routes.items():
        if len(route) > 1:
            color = colors[k % len(colors)]
            for i in range(len(route) - 1):
                u, v = route[i], route[i + 1]
                u_pos, v_pos = positions[u], positions[v]
                ax.plot([u_pos[0], v_pos[0]], [u_pos[1], v_pos[1]], 
                       color=color, linewidth=2, alpha=0.7, 
                       label=f'Vehicle {k}' if i == 0 else '')
                
                # Add arrow to show direction
                mid_x = (u_pos[0] + v_pos[0]) / 2
                mid_y = (u_pos[1] + v_pos[1]) / 2
                dx = v_pos[0] - u_pos[0]
                dy = v_pos[1] - u_pos[1]
                ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                           xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    
    # Plot assignment links (dashed lines from customers to their assigned depots)
    for customer, depot in assignments.items():
        if depot in opened_depots:
            cust_pos = positions[customer]
            depot_pos = positions[depot]
            ax.plot([cust_pos[0], depot_pos[0]], [cust_pos[1], depot_pos[1]], 
                   'k--', alpha=0.3, linewidth=1)
    
    ax.set_xlim(-1, 10)
    ax.set_ylim(0, 10)
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title(f'Location-Routing Problem Solution\nTotal Cost: ${total_cost:.2f}', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
if opened_depots and assignments and routes:
    visualize_lrp_solution(instance, opened_depots, assignments, routes)
else:
    print("No valid solution to visualize")

No valid solution to visualize


In [ ]:
def sensitivity_analysis(instance):
    """Perform sensitivity analysis on key parameters"""
    
    print("=" * 60)
    print("SENSITIVITY ANALYSIS")
    print("=" * 60)
    
    # 1. Vary depot costs
    print("\n1. Impact of Depot Cost Changes:")
    print("-" * 40)
    
    depot_cost_multipliers = [0.8, 1.0, 1.2, 1.5]
    results = []
    
    for multiplier in depot_cost_multipliers:
        # Create modified instance
        modified_instance = create_concrete_example()
        modified_instance.depot_costs = {j: cost * multiplier 
                                        for j, cost in instance.depot_costs.items()}
        
        # Solve
        prob, y_vars, z_vars, x_vars = solve_lrp_mip(modified_instance)
        opened_depots_mod, assignments_mod, routes_mod = extract_solution(
            modified_instance, prob, y_vars, z_vars, x_vars
        )
        
        if opened_depots_mod:
            total_cost_mod, fixed_cost_mod, routing_cost_mod = calculate_solution_cost(
                modified_instance, opened_depots_mod, assignments_mod, routes_mod
            )
            results.append({
                'Multiplier': multiplier,
                'Opened Depots': opened_depots_mod,
                'Fixed Cost': fixed_cost_mod,
                'Routing Cost': routing_cost_mod,
                'Total Cost': total_cost_mod
            })
    
    # Display results
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
    
    # 2. Vary vehicle capacity
    print("\n\n2. Impact of Vehicle Capacity Changes:")
    print("-" * 40)
    
    capacities = [25, 30, 40, 50]
    capacity_results = []
    
    for capacity in capacities:
        modified_instance = create_concrete_example()
        modified_instance.vehicle_capacity = capacity
        
        prob, y_vars, z_vars, x_vars = solve_lrp_mip(modified_instance)
        opened_depots_mod, assignments_mod, routes_mod = extract_solution(
            modified_instance, prob, y_vars, z_vars, x_vars
        )
        
        if opened_depots_mod:
            total_cost_mod, fixed_cost_mod, routing_cost_mod = calculate_solution_cost(
                modified_instance, opened_depots_mod, assignments_mod, routes_mod
            )
            num_vehicles_used = len(routes_mod)
            capacity_results.append({
                'Capacity': capacity,
                'Vehicles Used': num_vehicles_used,
                'Total Cost': total_cost_mod
            })
    
    df_capacity = pd.DataFrame(capacity_results)
    print(df_capacity.to_string(index=False))
    
    # Visualize sensitivity
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Depot cost impact
    ax1.plot(df_results['Multiplier'], df_results['Total Cost'], 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Depot Cost Multiplier')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Impact of Depot Cost Changes')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Capacity impact
    ax2.plot(df_capacity['Capacity'], df_capacity['Total Cost'], 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Vehicle Capacity')
    ax2.set_ylabel('Total Cost ($)')
    ax2.set_title('Impact of Vehicle Capacity Changes')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Perform sensitivity analysis
sensitivity_analysis(instance)

SENSITIVITY ANALYSIS

1. Impact of Depot Cost Changes:
----------------------------------------
Solving LRP using MIP...
Solving LRP using MIP...
Solving LRP using MIP...
Solving LRP using MIP...
 Multiplier Opened Depots  Fixed Cost  Routing Cost  Total Cost
        0.8           [4]        80.0             0        80.0
        1.0           [4]       100.0             0       100.0
        1.2           [4]       120.0             0       120.0
        1.5           [4]       150.0             0       150.0


2. Impact of Vehicle Capacity Changes:
----------------------------------------
Solving LRP using MIP...
Solving LRP using MIP...
Generation 10: Best fitness = 4101.11
Solving LRP using MIP...
Solving LRP using MIP...
 Capacity  Vehicles Used  Total Cost
       25              0         100
       30              0         100
       40              0         100
       50              0         100
✅ Swarm initialized: 20 particles
   Initial global best fitness: 57.15
Generat

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach. It provides:
- **Optimal solutions** with mathematical guarantees
- **Precise problem formulation** with all constraints explicitly modeled
- **Baseline for comparison** with all heuristic and advanced methods
- **Understanding of problem structure** through mathematical modeling

### Pros / Cons vs other approaches

**Pros:**
- ✅ **Guaranteed optimality** for small instances
- ✅ **Transparent decision process** - all constraints are visible
- ✅ **Reproducible results** - same input always gives same output
- ✅ **Foundation for understanding** the problem structure

**Cons:**
- ❌ **Scalability issues** - exponential time complexity
- ❌ **Computationally expensive** for large instances
- ❌ **Limited to small problems** in practice
- ❌ **Requires exact problem data** - no handling of uncertainty

### When to use this Tier
- **Small to medium instances** where optimality is important
- **Strategic planning** decisions where high costs justify thorough analysis
- **Benchmark development** to evaluate heuristic methods
- **Academic research** and teaching optimization concepts
- **Regulatory environments** requiring provable optimal solutions

### Key Insights from the Mathematical Formulation

1. **Trade-off Analysis**: The model explicitly balances fixed depot costs against variable routing costs

2. **Capacity Constraints**: Vehicle capacity limitations drive the need for multiple vehicles and affect depot location decisions

3. **Assignment Logic**: Customer-depot assignments are endogenous to the optimization - they're decided alongside routing

4. **Computational Complexity**: The interaction between location and routing decisions creates a highly complex combinatorial problem

This mathematical foundation is essential for understanding the Location-Routing Problem and serves as the benchmark against which all advanced methods will be compared.